# NFL Public Data Warehouse — Collection & Exploration

**Goal:** Collect publicly accessible NFL data into separate pandas DataFrames, inventory source coverage, and export results for analysis.

Review each source's terms, licenses, rate limits, and attribution requirements before publishing or redistributing data.

## Scope and limits
- Core bulk collection uses **nflreadpy / nflverse**, which offers maintained downloadable datasets.
- NFL player data from 1999.
- An inventory cell queries GitHub's public API for the complete current nflverse release/asset catalog.
- An optional ESPN scoreboard example demonstrates a public endpoint, but unofficial endpoints can change without notice.
- Official NFL pages, Pro Football Reference, Kaggle, GitHub projects, college-football sources, and licensed providers are listed in the source registry for follow-up.
- Raw RFID/player tracking at 10 Hz, complete All-22 video, private team data, some charting data, and some authenticated NFL endpoints are **not** all publicly downloadable. Public Next Gen Stats summaries are not the same as raw tracking data.

The notebook loads broad historical coverage by default where supported. For large datasets, run one category at a time or change `SEASONS`.

In [6]:
from pathlib import Path
import nflreadpy as nfl
import pandas as pd

In [8]:
# Historical coverage: 2000 through the 2026 season.
# 2026 will contain only games available from the source.
SEASONS = list(range(2000, 2027))
POSITIONS = {"QB", "WR", "TE", "RB", "K"}

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load player-game statistics and convert to pandas.
stats = nfl.load_player_stats(SEASONS).to_pandas()

# Normalize column names used by different source versions.
rename_map = {
    "gsis_id": "player_id",
    "recent_team": "team",
}

for source, target in rename_map.items():
    if source in stats.columns and target not in stats.columns:
        stats = stats.rename(columns={source: target})

# Retain the positions you want.
stats["position"] = stats["position"].astype("string").str.upper()
stats = stats.loc[stats["position"].isin(POSITIONS)].copy()

# Normalize team abbreviations and categorical values.
for col in ["team", "opponent_team"]:
    if col in stats.columns:
        stats[col] = stats[col].astype("string").str.upper()

# Retain only valid player records.
stats = stats.loc[stats["player_id"].notna()].copy()

# Build a stable ordering key.
# game_id is important because week alone cannot distinguish
# all games and postseason situations.
sort_cols = [
    c for c in
    ["season", "week", "game_id", "player_id"]
    if c in stats.columns
]
stats = stats.sort_values(sort_cols).reset_index(drop=True)

# Do not silently remove duplicate player-game records.
key = [
    c for c in
    ["player_id", "season", "season_type", "game_id"]
    if c in stats.columns
]
duplicates = stats.loc[stats.duplicated(key, keep=False)] if key else pd.DataFrame()

if not duplicates.empty:
    print("Review duplicate player-game keys:", len(duplicates))

# Preserve the full source schema, including extra columns
# supplied by future data releases.
stats.to_parquet(
    OUTPUT_DIR / "player_game_stats.parquet",
    index=False
)

stats.to_csv(
    OUTPUT_DIR / "player_game_stats.csv",
    index=False
)

print(f"Rows: {len(stats):,}")
print(f"Columns: {len(stats.columns):,}")
print(stats.groupby("season").size().tail(10))
print(stats.head())

Rows: 161,056
Columns: 150
season
2017    6209
2018    6121
2019    6165
2020    6354
2021    6702
2022    6653
2023    6640
2024    6710
2025    6890
2026    1959
dtype: int64
    player_id  player_name player_display_name position position_group  \
0  00-0000745     T.Barber         Tiki Barber       RB             RB   
1  00-0001343  C.Blanchard      Cary Blanchard        K           SPEC   
2  00-0001532     D.Boston        David Boston       WR             WR   
3  00-0002565   D.Campbell        Dan Campbell       TE             TE   
4  00-0003175       M.Cody            Mac Cody       WR             WR   

                                        headshot_url  season  week  \
0  https://static.www.nfl.com/image/private/f_aut...    2000     1   
1  https://static.www.nfl.com/image/private/f_aut...    2000     1   
2  https://static.www.nfl.com/image/private/f_aut...    2000     1   
3  https://static.www.nfl.com/image/private/f_aut...    2000     1   
4  https://static.www.nfl.co

In [12]:
stats.columns.tolist()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'game_id',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_

In [ ]:
schedule = nfl.load_schedules(SEASONS).to_pandas()

schedule_cols = [
    c for c in [
        "game_id", "gameday", "home_team", "away_team",
        "home_score", "away_score", "season_type"
    ]
    if c in schedule.columns
]

schedule = schedule[schedule_cols].drop_duplicates("game_id")

games = stats.merge(
    schedule,
    on="game_id",
    how="left",
    suffixes=("", "_schedule"),
    validate="many_to_one",
)

if "gameday" in games.columns:
    games["game_date"] = pd.to_datetime(
        games["gameday"], errors="coerce"
    )

if {"team", "home_team", "away_team"}.issubset(games.columns):
    games["is_home"] = (
        games["team"] == games["home_team"]
    ).astype("int8")

    games["opponent_team"] = games["team"].map(
        lambda team: pd.NA
    )  # Replace below using the opponent's game context.

In [ ]:
import numpy as np

if {"team", "home_team", "away_team"}.issubset(games.columns):
    is_home = games["team"].eq(games["home_team"])
    is_away = games["team"].eq(games["away_team"])

    games["is_home"] = is_home.astype("int8")
    games["opponent_team"] = np.where(
        is_home,
        games["away_team"],
        np.where(is_away, games["home_team"], pd.NA),
    )

games.to_parquet(
    OUTPUT_DIR / "player_game_stats_with_context.parquet",
    index=False,
)

In [ ]:
games = games.sort_values(
    ["player_id", "game_date", "game_id"]
).copy()

FEATURE_STATS = [
    "fantasy_points_ppr",
    "passing_yards",
    "rushing_yards",
    "receiving_yards",
    "targets",
    "receptions",
    "carries",
]

for stat in FEATURE_STATS:
    if stat not in games.columns:
        continue

    for window in [3, 5]:
        feature = f"{stat}_avg_last_{window}"

        games[feature] = (
            games.groupby("player_id", sort=False)[stat]
            .transform(
                lambda s: s.shift(1).rolling(
                    window=window,
                    min_periods=1,
                ).mean()
            )
        )

    games[f"{stat}_previous_game"] = (
        games.groupby("player_id", sort=False)[stat]
        .shift(1)
    )

# Save the feature dataset separately from raw statistics.
games.to_parquet(
    OUTPUT_DIR / "player_game_features.parquet",
    index=False,
)

In [1]:
import nflreadpy as nfl

# Load current season play-by-play data
pbp = nfl.load_pbp()

# Load player game-level stats for multiple seasons
player_stats = nfl.load_player_stats([2022, 2023])

# Load all available team level stats
team_stats = nfl.load_team_stats(seasons=True)

# nflreadpy uses Polars instead of pandas. Convert to pandas if needed:
pbp_pandas = pbp.to_pandas()

In [3]:
player_stats

player_id,player_name,player_display_name,position,position_group,headshot_url,season,week,season_type,game_id,team,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,pacr,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,…,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
str,str,str,str,str,str,i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64
"""00-0019596""","""T.Brady""","""Tom Brady""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2022,1,"""REG""","""2022_01_TB_DAL""","""TB""","""DAL""",18,27,212,1,1,2,-17,0,0,305,87,9,1.050854,3.290235,0,0.695082,7,5,3,1,2,-1,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,10.38,10.38
"""00-0022824""","""A.Lee""","""Andy Lee""","""P""","""SPEC""","""https://static.www.nfl.com/ima…",2022,1,"""REG""","""2022_01_KC_ARI""","""ARI""","""KC""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,5,0,57,255,0,0,0,0,0,5,44,0,211,0.0,0.0
"""00-0023252""","""R.Gould""","""Robbie Gould""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2022,1,"""REG""","""2022_01_SF_CHI""","""SF""","""CHI""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,"""25""",null,null,25,0,0,1,1,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0023459""","""A.Rodgers""","""Aaron Rodgers""","""QB""","""QB""","""https://static.www.nfl.com/ima…",2022,1,"""REG""","""2022_01_GB_MIN""","""GB""","""MIN""",22,34,195,0,1,4,-33,1,1,190,134,9,-11.627545,-2.784771,0,1.026316,5,4,3,0,1,-1,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,3.7,3.7
"""00-0023853""","""M.Prater""","""Matt Prater""","""K""","""SPEC""","""https://static.www.nfl.com/ima…",2022,1,"""REG""","""2022_01_KC_ARI""","""ARI""","""KC""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,1,1,0,0,1.0,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""00-0038643""","""B.Willis""","""Brayden Willis""","""TE""","""TE""","""https://static.www.nfl.com/ima…",2023,22,"""POST""","""2023_22_SF_KC""","""SF""","""KC""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0038982""","""C.Conner""","""Chamarri Conner""","""DB""","""DB""","""https://static.www.nfl.com/ima…",2023,22,"""POST""","""2023_22_SF_KC""","""KC""","""SF""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0,0,0,0,0.0,0.0
"""00-0039006""","""F.Anudike-Uzomah""","""Felix Anudike-Uzomah""","""DE""","""DL""","""https://static.www.nfl.com/ima…",2023,22,"""POST""","""2023_22_SF_KC""","""KC""","""SF""",0,0,0,0,0,0,0,0,0,0,0,0,null,null,0,null,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,null,null,null,0,0,0,0,0,0,0,null,0,0,0,0,0,0,0

In [4]:
team_stats

season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,passing_interceptions,sacks_suffered,sack_yards_lost,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_cpoe,passing_2pt_conversions,passing_10,passing_16,passing_20,passing_40,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,rushing_10,rushing_12,rushing_20,rushing_40,…,fg_made_50_59,fg_made_60_,fg_missed_0_19,fg_missed_20_29,fg_missed_30_39,fg_missed_40_49,fg_missed_50_59,fg_missed_60_,fg_made_list,fg_missed_list,fg_blocked_list,fg_made_distance,fg_missed_distance,fg_blocked_distance,pat_made,pat_att,pat_missed,pat_blocked,pat_pct,gwfg_made,gwfg_att,gwfg_missed,gwfg_blocked,gwfg_distance,pt_att,pt_blocked,pt_long,pt_yards,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards
i32,i32,str,str,str,str,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,str,str,str,i32,i32,i32,i32,i32,i32,i32,f64,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
1999,1,"""ARI""","""REG""","""1999_01_ARI_PHI""","""PHI""",25,48,274,1,3,1,-8,0,0,32,274,15,-1.911235,null,0,13,4,2,1,36,78,1,1,1,6,-10.060235,0,3,2,0,0,…,0,0,0,0,0,0,0,0,"""31;25;32;31""",null,null,119,0,0,1,1,0,0,1.0,1,1,0,0,31,5,0,40,172,0,1,0,0,3,1,2,0,170
1999,1,"""ATL""","""REG""","""1999_01_MIN_ATL""","""MIN""",18,31,290,1,0,2,-12,1,1,0,290,16,12.035272,null,0,13,8,6,1,23,81,1,0,0,3,-5.527196,0,2,2,1,0,…,0,0,0,0,2,0,0,0,null,"""35;39""",null,0,74,0,2,2,0,0,1.0,0,0,0,0,0,2,0,48,88,1,0,1,0,0,1,-1,0,89
1999,1,"""BUF""","""REG""","""1999_01_BUF_IND""","""IND""",22,42,300,1,2,5,-26,1,1,20,300,15,-10.926169,null,0,12,6,6,1,15,47,0,0,0,1,-1.781102,1,0,0,0,0,…,0,0,0,0,0,0,0,0,"""36;29""",null,null,65,0,0,0,0,0,0,null,0,0,0,0,0,6,0,44,206,0,0,1,2,3,0,0,0,166
1999,1,"""CAR""","""REG""","""1999_01_CAR_NO""","""NO""",16,32,207,1,1,7,-46,1,1,4,207,9,-17.287012,null,0,8,4,4,0,16,85,0,0,0,3,-1.15562,0,3,2,2,0,…,1,0,0,0,0,0,1,0,"""52""","""53""",null,52,53,0,1,1,0,0,1.0,0,0,0,0,0,8,0,51,355,1,0,1,0,2,5,31,0,324
1999,1,"""CHI""","""REG""","""1999_01_KC_CHI""","""KC""",31,47,322,2,0,1,-15,1,1,0,322,16,-4.002322,null,0,14,7,5,0,28,81,0,0,0,6,-6.414171,0,2,1,0,0,…,0,0,0,0,0,0,0,0,"""21;24""",null,null,45,0,0,2,2,0,0,1.0,0,0,0,0,0,5,0,47,183,3,1,1,0,1,2,7,0,176
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026,5,"""TB""","""REG""","""2026_05_TB_DAL""","""DAL""",19,25,189,1,0,1,-11,0,0,109,132,10,5.87207,2.793341,0,7,3,2,1,39,241,2,0,0,10,4.308134,0,5,3,2,1,…,0,0,0,0,0,0,0,0,"""41""",null,null,41,0,0,3,3,0,0,1.0,0,0,0,0,0,4,0,49,163,2,0,1,0,2,1,10,0,153
2026,5,"""TEN""","""REG""","""2026_05_HOU_TEN""","""HOU""",23,36,170,0,1,4,-40,0,0,293,100,8,-16.936315,-2.365735,0,8,3,1,0,18,57,0,1,1,5,-5.563077,0,1,1,0,0,…,0,0,0,0,0,0,0,0,"""30""",null,null,30,0,0,0,0,0,0,null,0,0,0,0,0,4,0,56,188,1,0,0,0,2,2,21,0,167
2026,5,"""WAS""","""REG""","""2026_05_NYG_WAS""","""NYG""",17,25,199,0,0,5,-23,0,0,201,103,11,-2.388523,8.076239,1,11,5,3,0,30,202,3,0,0,8,9.048185,0,6,5,4,0,…,0,0,0,0,0,0,0,0,"""28;40""",null,null,68,0,0,1,2,1,0,0.5,0,0,0,0,0,6,0,59,312,2,0,1,0,1,4,52,0,260


In [2]:
# Load player stats
load_player_stats(
    seasons: int | list[int] | bool | None = None,
    summary_level: Literal[
        "week", "reg", "post", "reg+post"
    ] = "week",
) -> pl.DataFrame

SyntaxError: invalid syntax (1215091628.py, line 3)